### Setup & configuration

In [1]:
import sys
import importlib
import json
import time
import mlflow
import pandas as pd
import torch
from IPython.display import display, Markdown, JSON, HTML

#
sys.path.append("..")

/media/zirox2025/DATA/running/GitHub/Knowledge-base-Extraction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### INPUT VARIABLES

In [2]:
WORKLOG = """
INC-99102

System monitoring detected intermittent packet loss on core switch cluster in Toronto data node TD-14.

Initial alert triggered at 03:41 AM when ICMP ping failures exceeded threshold (packet loss > 40%) for IP range 10.22.14.0/24.

NOC engineer report:
Multiple devices intermittently responding. Device SW-CORE-19 shows unstable connectivity. SSH access failed twice.

At 03:58 AM, logs show repeated ICMP timeout for 10.22.14.23 and 10.22.14.31.

A technician was dispatched but initial remote diagnostics were inconclusive due to network flapping.

On-site technician report (Alex P.):
Device SW-CORE-19 completely unresponsive to ping and console access.
Physical inspection revealed NIC card failure suspected after overheating warning logs.

Replacement procedure initiated:
- removed faulty network interface card
- installed replacement NIC module
- rebooted device

By 05:10 AM, device restored and stable ping response confirmed.

Customer impact: intermittent service degradation across VLAN 220-245.

Next action: monitor stability for 24 hours and confirm no packet loss recurrence.

"""

# Knowledge Extraction

### setup MLflow

In [3]:
import mlflow

mlflow.set_experiment("incident-log-extraction-qwen")

2026/06/06 13:30:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/06 13:30:51 INFO mlflow.store.db.utils: Updating database tables
2026/06/06 13:31:30 INFO mlflow.tracking.fluent: Experiment with name 'incident-log-extraction-qwen' does not exist. Creating a new experiment.


<Experiment: artifact_location='/media/zirox2025/DATA/running/GitHub/Knowledge-base-Extraction/notebooks/mlruns/1', creation_time=1780767090323, experiment_id='1', last_update_time=1780767090323, lifecycle_stage='active', name='incident-log-extraction-qwen', tags={}, trace_location=None, workspace='default'>

### Instantiate the LLM model

In [ ]:
MODEL_NAME = "../model/Qwen2.5-1.5B-Instruct"

import json
import time
import os
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

# define the absolute path of the LLM model
MODEL_PATH = os.path.abspath(MODEL_NAME)
print(f"Loading model {MODEL_PATH}...")

# instantiate the model and tokenizer with local_files_only=True to ensure it loads from the local path
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

model.eval()

Loading model /media/zirox2025/DATA/running/GitHub/Knowledge-base-Extraction/model/Qwen2.5-1.5B-Instruct...


### Run the experiment

In [ ]:
import importlib
import prompts

importlib.reload(prompts)

import main

importlib.reload(main)

from typing import Any, Dict, Tuple

# load modules/prompts
from prompts import PROMPT_V2 as prompt_template
from main import load_model_and_tokenizer, build_structured_generator, extract_incident, verify_extraction, evaluate_extraction

print("KNOWLEDGE EXTRACTION PIPELINE")
print("=" * 80)

model_name = "../model/Qwen2.5-1.5B-Instruct"

worklog = """
INC-99102

System monitoring detected intermittent packet loss on core switch cluster in Toronto data node TD-14.

Initial alert triggered at 03:41 AM when ICMP ping failures exceeded threshold (packet loss > 40%) for IP range 10.22.14.0/24.

NOC engineer report:
Device SW-CORE-19 unstable. SSH access failed.

At 03:58 AM, ICMP timeout for 10.22.14.23 and 10.22.14.31.

Technician Alex P dispatched.

NIC failure suspected after overheating logs.

NIC replaced and device rebooted.

Next action: monitor stability for 24 hours.
"""

mlflow.set_experiment("knowledge_extraction_pipeline")

with mlflow.start_run():
    start_time = time.time()

    print("[STEP] Initializing pipeline...")

    tokenizer, model = load_model_and_tokenizer(
        model_name=model_name,
    )

    structured_model = build_structured_generator(
        model=model,
        tokenizer=tokenizer,
    )

    extraction = extract_incident(
        structured_model=structured_model,
        worklog=worklog,
    )

    verification = verify_extraction(
        model=model,
        tokenizer=tokenizer,
        worklog=worklog,
        extracted=extraction,
    )

    metrics = evaluate_extraction(
        extraction=extraction,
    )

    metrics["latency_sec"] = time.time() - start_time

    print("[STEP] Logging MLflow metrics...")

    mlflow.log_param(
        "model_name",
        model_name,
    )

    for key, value in metrics.items():
        mlflow.log_metric(key, value)

    mlflow.log_text(
        worklog,
        "input_worklog.txt",
    )

    mlflow.log_text(
        extraction.model_dump_json(indent=2),
        "extraction.json",
    )

    mlflow.log_text(
        verification.model_dump_json(indent=2),
        "verification.json",
    )

    print("[DONE] MLflow logging complete.")

    print("\n" + "=" * 80)
    print("FINAL EXTRACTION")
    print("=" * 80)

    print(
        extraction.model_dump_json(
            indent=2,
        )
    )

    print("\n" + "=" * 80)
    print("VERIFICATION")
    print("=" * 80)

    print(
        verification.model_dump_json(
            indent=2,
        )
    )

    print("\n[DONE] Pipeline completed successfully.")

In [ ]:
extraction

In [ ]:
display(JSON(extraction.model_dump_json(indent=2)))

In [ ]:
import importlib
import prompts

importlib.reload(prompts)

import main

importlib.reload(main)

from typing import Any, Dict, Tuple

# load modules/prompts
from main import load_model_and_tokenizer, build_structured_generator, extract_incident, verify_extraction, evaluate_extraction

verification = verify_extraction(
    model=model,
    tokenizer=tokenizer,
    worklog=worklog,
    extracted=extraction,
)
display(JSON(verification.model_dump_json(indent=2)))

[STEP] Running verification...
[DONE] Verification complete. (196.96s)


<IPython.core.display.JSON object>

In [ ]:
import time
import mlflow
import torch
from typing import Dict

import outlines
from transformers import AutoModelForCausalLM, AutoTokenizer

from pydantic import BaseModel, Field, ConfigDict
from typing import List, Optional, Dict as TypingDict


import importlib
import prompts

importlib.reload(prompts)

import main

importlib.reload(main)

from typing import Any, Dict, Tuple

# load modules/prompts
from prompts import PROMPT_V2 as prompt_template
from main import load_model_and_tokenizer, build_structured_generator, extract_incident, verify_extraction, evaluate_extraction

# =========================================================
# PYDANTIC SCHEMAS (SOURCE OF TRUTH)
# =========================================================


class Event(BaseModel):
    timestamp: Optional[str] = None
    description: str
    actor: Optional[str] = None
    system: Optional[str] = None


class Entity(BaseModel):
    name: str
    type: str
    metadata: Optional[TypingDict[str, str]] = None


class IncidentExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    incident_id: str
    incident_summary: str
    event_timeline: List[Event]
    entities: TypingDict[str, Entity]
    final_resolution: str


class VerificationIssue(BaseModel):
    """Single verification issue."""

    issue_type: str
    details: str


class VerificationResult(BaseModel):
    """Verification output schema."""

    fix_required: bool
    issues: List[VerificationIssue] = Field(default_factory=list)


# =========================================================
# OUTLINES CONSTRAINED DECODER (ZERO-FAILURE CORE)
# =========================================================

generator = outlines.from_transformers(model, tokenizer)


def incident_generator(generator, prompt: str):
    return generator(
        prompt,
        IncidentExtraction,
        max_new_tokens=384,
        temperature=0.1,
    )


# =========================================================
# ENRICHMENT (OPTIONAL, NOT REQUIRED FOR CORRECTNESS)
# =========================================================


def enrich(generator, worklog: str, extracted: IncidentExtraction, verification: VerificationResult):

    if not verification.fix_required:
        return extracted

    prompt = f"""
Improve extracted incident ONLY if missing critical information exists.

WORKLOG:
{worklog}

CURRENT EXTRACTION:
{extracted.model_dump_json(indent=2)}

VERIFICATION:
{verification.model_dump_json(indent=2)}

If verification reports issues, update the extraction to fix them.
If verification reports no issues, return the current extraction unchanged.
"""

    output = incident_generator(generator, prompt)

    # print("[DEBUG] Verification raw output:")
    # print(decoded)
    # json_payload = extract_json_payload(decoded)

    print("[DEBUG] enrichment raw output:")
    print(output)

    try:
        return IncidentExtraction.model_validate(output).model_dump_json(indent=2)
    except Exception as e:
        print(f"[ERROR] Enrichment failed to produce valid output: {e}")
        print("Returning the raw enrichment output.")
        return output
        # print("Returning original extraction without enrichment.")
        # return extracted


# 3. optional enrichment (only if needed)
final_output = enrich(generator, WORKLOG, extraction, verification)
display(JSON(final_output.model_dump_json(indent=2)))

<IPython.core.display.JSON object>

In [ ]:
display(JSON(extraction.model_dump_json(indent=2)))

<IPython.core.display.JSON object>

In [ ]:
display(JSON(verification.model_dump_json(indent=2)))

<IPython.core.display.JSON object>

# Syntax